# Phase 4 — CJPE · Experiment 3: Phase-1 **SWA decoder** + **LoRMA$_+$** (Colab) — single 4-epoch run

> **Single-run version.** One training run of **4 epochs** on the training set only. No Stage B (no retraining on train + dev), no A/B ensemble, no extra runs. The dev set is used only for monitoring (evaluated every half epoch) and for the positive rate that sets the final threshold. The submission file comes from this one run's weights.

**Model:** the Phase-1 **local/global sliding-window causal decoder** (`best_model_swa.pt`, 16 layers, d=768, 12 query / 4 KV heads, RoPE, SwiGLU, tied embeddings, local window 128 with global attention at layers 4/8/12/16, context **1024**) + the same classification head as the GQA-decoder run. The backbone is **frozen**; only **LoRMA$_+$** factors and the head are trained.

Everything else is copied from `phase4_exp2_gqa_decoder_full_ft`: lower-cased, space-token-removed v2 token cache (reused from Drive), document-level attention over chunks with a distance-from-end embedding plus the last chunk's vector, last-token + mean pooling into a LayerNorm, class-weighted label-smoothed loss, residual dropout / chunk dropout / weight decay 0.05, **EMA weights**, dev every half epoch, **one** 4-epoch run on the training set (dev for monitoring only), final = that run's weights with the top 50% of the test scores → class 1, 14 GB memory cap.

**The two changes**

| GQA-decoder run | this run | why |
|---|---|---|
| dense causal GQA (`is_causal=True`, SDPA), context **512**, window = `</s>` + 510 + `</s>`, `K = 8` (8 × 510 = 4,080 compact tokens) | Phase-1 **sliding-window** attention verbatim (chunked local + 4 global layers), context **1024**, window = `</s>` + **1022** + `</s>`, `K = 4` (4 × 1022 = **4,088** compact tokens) | the SWA checkpoint has `max_seq_len = 1024`; the per-document token budget is held fixed, so the v2 cache is unchanged |
| **full** fine-tuning (all 139M weights) | **LoRMA$_+$**, written from scratch (Assignment-4 Q4.4), no `peft` / `loralib` / any LoRA library | the frozen backbone is adapted **multiplicatively** |

**LoRMA$_+$ (not LoRA).** LoRA *adds* a low-rank matrix; LoRMA$_+$ *multiplies* by a low-rank-parameterised matrix with additive rank inflation:

$$W' = \left(I + \tfrac{\alpha}{r}BA\right)W_0 ,\qquad A\in\mathbb{R}^{r\times d_\text{out}},\; B\in\mathbb{R}^{d_\text{out}\times r}$$

applied in the **re-ordered** form so no $d_\text{out}\times d_\text{out}$ product is ever materialised: $y = xW_0^\top$, then $y \leftarrow y + \tfrac{\alpha}{r}\,y A^\top B^\top$. With $B=0$ at initialisation the transformation is exactly the identity, so the adapted model starts out equal to Phase 1.

**Memory:** capped at 14 GB; the peak is asserted at the end. `best_model_swa.pt` holds optimiser + RNG state, so a weights-only copy is extracted to Drive once.

## 1. Drive, paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import glob
from pathlib import Path

# ---- EDIT IF NEEDED -------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive"
CKPT_HINT  = f"{DRIVE_ROOT}/best_model_swa.pt"      # Phase-1 SWA decoder
TOK_HINT   = f"{DRIVE_ROOT}/legal_bpe_a1_50k_final.json"
V2_DIR     = Path(f"{DRIVE_ROOT}/phase4_exp1_v2_encoder_full_ft")   # token cache is reused from here
OUT_DIR    = Path(f"{DRIVE_ROOT}/phase4_exp3_swa_decoder_lorma_plus")
# ---------------------------------------------------------------------------

def find_in_drive(hint, name):
    if Path(hint).exists():
        return Path(hint)
    hits = glob.glob(f"{DRIVE_ROOT}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found in Drive. Set the path above.")
    return Path(hits[0])

CKPT_DRIVE = find_in_drive(CKPT_HINT, "best_model_swa.pt")
TOK_PATH   = find_in_drive(TOK_HINT,  "legal_bpe_a1_50k_final.json")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Phase-1 SWA decoder checkpoint:", CKPT_DRIVE)
print("Phase-1 tokenizer :", TOK_PATH)
print("Output directory  :", OUT_DIR)

Mounted at /content/drive
Phase-1 SWA decoder checkpoint: /content/drive/MyDrive/best_model_swa.pt
Phase-1 tokenizer : /content/drive/MyDrive/legal_bpe_a1_50k_final.json
Output directory  : /content/drive/MyDrive/phase4_exp3_swa_decoder_lorma_plus


## 2. Imports, memory cap, hyper-parameters

In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import sys, json, math, random, gc, re, shutil, heapq, copy
import multiprocessing as mp
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Runtime -> Change runtime type -> GPU.")
device = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# ---- hard memory cap: behave like a free 15 GB T4 on any GPU ----
MEM_CAP_GB = 14.0
if gpu_gb > MEM_CAP_GB + 0.5:
    torch.cuda.set_per_process_memory_fraction(MEM_CAP_GB / gpu_gb)

if torch.cuda.is_bf16_supported():
    AMP_DTYPE, USE_SCALER = torch.bfloat16, False
else:                                   # T4: fp16 + dynamic loss scaling
    AMP_DTYPE, USE_SCALER = torch.float16, True
GRAD_CKPT = True                        # Phase-1 SWA attention builds its score matrices explicitly

# ---- data / window constants (1024-token backbone instead of 512) ----
SPLITS = ["train", "validation", "test"]
BODY, WIN = 1022, 1024                  # body tokens per window ('</s>' + 1022 + '</s>' = 1024)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPACE_ID = 36
K = 4                                   # chunks per document (4 x 1022 = 4,088 compact tokens)
TAIL_TOK = 4096                         # compact tokens kept per document (last ones)
TAIL_CH = 28000                         # characters tokenised from the end of long documents

# ---- batching ----
EFF_DOCS = 32                           # documents per optimiser step (larger batch for G4 throughput)
MICRO_WINDOWS = 16                      # windows per training micro-batch (1024 tokens each; larger G4 micro-batch)
EVAL_WINDOWS = 16                       # windows per inference batch

# ---- LoRMA+ (Assignment-4 Q4.4 implementation; no external PEFT library) ----
LORMA_R, LORMA_ALPHA = 8, 16
LORMA_TARGETS = ("q_proj", "k_proj", "v_proj", "out_proj")   # attention projections
LORMA_SWAP_AB = False                   # False -> B = 0 at init (paper init for LoRMA+); True -> A = 0
LORMA_LR, HEAD_LR, LLRD = 5e-4, 5e-4, 0.95
LORMA_WD = 0.0                          # no weight decay on the low-rank factors

# ---- optimisation (unchanged from the GQA-decoder full-FT run apart from the LRs) ----
WD, WARMUP, CLIP = 0.05, 0.06, 1.0
E_MAX, DPE = 4, 16000                   # 4 epochs in one single run; documents drawn per epoch
EVAL_EVERY = 250                        # steps between dev evaluations (half an epoch with 500 steps/epoch)
EMA_DECAY = 0.998
LABEL_SMOOTH = 0.05
RESID_DROPOUT, HEAD_DROPOUT, CHUNK_DROP = 0.10, 0.10, 0.10
TARGET_POS_RATE = None                  # None -> use the dev positive rate (0.50)
SUBMIT_WEIGHTS = "final"                # "final": EMA weights after the 4th epoch | "best": EMA weights at the best-dev-AUC evaluation
assert SUBMIT_WEIGHTS in ("final", "best")

print("gpu            :", gpu_name, f"({gpu_gb:.0f} GB) | enforced cap {MEM_CAP_GB} GB")
print("autocast dtype :", str(AMP_DTYPE).replace("torch.", ""), "| loss scaling:", USE_SCALER)
print("windows        :", f"K={K} x {WIN} tokens ({K*BODY} compact tokens per document)")
print("LoRMA+         :", f"r={LORMA_R}, alpha={LORMA_ALPHA}, targets={LORMA_TARGETS}, swap_ab={LORMA_SWAP_AB}")
print("submission from:", f"{SUBMIT_WEIGHTS} weights of one {E_MAX}-epoch run (train only, no Stage B, no ensemble)")
print("steps/epoch    :", DPE // EFF_DOCS, "| total steps:", E_MAX * (DPE // EFF_DOCS))

gpu            : NVIDIA RTX PRO 6000 Blackwell Server Edition (95 GB) | enforced cap 14.0 GB
autocast dtype : bfloat16 | loss scaling: False
windows        : K=4 x 1024 tokens (4088 compact tokens per document)
LoRMA+         : r=8, alpha=16, targets=('q_proj', 'k_proj', 'v_proj', 'out_proj'), swap_ab=False
submission from: final weights of one 4-epoch run (train only, no Stage B, no ensemble)
steps/epoch    : 500 | total steps: 2000


## 3. Phase-1 tokenizer (unchanged)

In [3]:
N_SPECIAL, N_BYTES = 4, 256
BASE_VOCAB_SIZE = N_SPECIAL + N_BYTES
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

class A1Tokenizer:
    def __init__(self, itos, merges):
        self.itos = list(itos)
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos)
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}
        self._id_to_bytes = [b""] * N_SPECIAL + [bytes([b]) for b in range(256)]
        for l, r in self.merges:
            self._id_to_bytes.append(self._id_to_bytes[l] + self._id_to_bytes[r])
        assert len(self._id_to_bytes) == self.vocab_size
        self._cache = {}

    def _encode_word(self, word):                       # identical to Phase 1
        raw = word.encode("utf-8")
        seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks:
            return seq
        n = len(seq)
        prev = [-1] + list(range(0, n - 1))
        nxt = list(range(1, n)) + [-1]
        active = [True] * n
        heap = []
        for i in range(n - 1):
            rank = self._merge_ranks.get((seq[i], seq[i + 1]))
            if rank is not None:
                heapq.heappush(heap, (rank, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]:
                continue
            if nxt[left] != right or prev[right] != left:
                continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank:
                continue
            seq[left] = BASE_VOCAB_SIZE + rank
            active[right] = False
            right_next = nxt[right]
            nxt[left] = right_next
            if right_next != -1:
                prev[right_next] = left
            left_prev = prev[left]
            if left_prev != -1:
                new_rank = self._merge_ranks.get((seq[left_prev], seq[left]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left_prev, left))
            if right_next != -1:
                new_rank = self._merge_ranks.get((seq[left], seq[right_next]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left, right_next))
        out, i = [], 0
        while i != -1:
            if active[i]:
                out.append(seq[i])
            i = nxt[i]
        return out

    def encode_normalized(self, text):
        # text: single-space separated, no leading/trailing space
        out, cache, first = [], self._cache, True
        for w in text.split(" "):
            if not first:
                out.append(N_SPECIAL + 32)
            first = False
            ids = cache.get(w)
            if ids is None:
                ids = self._encode_word(w)
                if len(cache) > 1_500_000:
                    cache.clear()
                cache[w] = ids
            out.extend(ids)
        return out

    def decode(self, ids):
        buf, chunks = bytearray(), []
        for i in ids:
            i = int(i)
            if i < N_SPECIAL:
                chunks.append(bytes(buf).decode("utf-8", errors="replace")); buf.clear()
                chunks.append(SPECIAL_TOKENS[i])
            else:
                buf.extend(self._id_to_bytes[i])
        chunks.append(bytes(buf).decode("utf-8", errors="replace"))
        return "".join(chunks)

with open(TOK_PATH, "r", encoding="utf-8") as f:
    _payload = json.load(f)
TOK = A1Tokenizer(_payload["itos"], _payload["merges"])
assert TOK.vocab_size == 50000 and TOK.itos[:4] == SPECIAL_TOKENS
assert N_SPECIAL + 32 == SPACE_ID

_t = "The Supreme Court held that the appeal was dismissed. भारत के संविधान के अनुसार।"
_ids = TOK.encode_normalized(_t)
assert TOK.decode(_ids) == _t
print("tokenizer ready | vocab", TOK.vocab_size, "| round-trip OK | space token id", SPACE_ID)

tokenizer ready | vocab 50000 | round-trip OK | space token id 36


## 4. Token cache v2 (lower-cased, compact, last 4,096 tokens)

Identical to the earlier runs and independent of the window length, so the `v2_lower_compact_t4096_ch28000` files written by Experiment 1 are read straight from Drive. Test labels are never read.

In [4]:
SIG = f"v2_lower_compact_t{TAIL_TOK}_ch{TAIL_CH}"
CACHE_DIR = V2_DIR / "token_cache" if (V2_DIR / "token_cache").exists() else OUT_DIR / "token_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
def cache_path(split): return CACHE_DIR / f"{split}_{SIG}.npz"
CACHE_READY = all(cache_path(s).exists() for s in SPLITS)
print("v2 token caches on Drive:", CACHE_READY, "|", CACHE_DIR)

_TEXTS = []

def _tok_job(i):
    raw = _TEXTS[i]
    no_upper = not any(c.isupper() for c in raw[:3000])
    s = " ".join(raw.lower().split()) or "."
    n = len(s)
    if n > TAIL_CH:
        s = s[-TAIL_CH:]
        cut = s.find(" ")
        s = s[cut + 1:] if cut >= 0 else s
    ids = np.asarray(TOK.encode_normalized(s), np.int64)
    ids = ids[ids != SPACE_ID]                        # compact representation
    full = (n <= TAIL_CH) and len(ids) <= TAIL_TOK
    return (ids[-TAIL_TOK:].astype(np.uint16), full, n, " ".join(raw.split())[:300], no_upper)

def tokenize_texts(texts):
    global _TEXTS
    _TEXTS = texts
    ctx = mp.get_context("fork")
    out, step = [], max(1, len(texts) // 10)
    with ctx.Pool(os.cpu_count()) as pool:
        for k, r in enumerate(pool.imap(_tok_job, range(len(texts)), chunksize=16), 1):
            out.append(r)
            if k % step == 0 or k == len(texts):
                print(f"   tokenised {k:,}/{len(texts):,}")
    _TEXTS = []
    return out

def load_json_records(path):
    with open(path, "r", encoding="utf-8") as f:
        first = f.read(1); f.seek(0)
        if first == "[":
            return json.load(f)
        return [json.loads(l) for l in f if l.strip()]

if not CACHE_READY:
    from huggingface_hub import hf_hub_download, login
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    if not HF_TOKEN:
        import getpass
        HF_TOKEN = getpass.getpass("Hugging Face token (read access): ")
    login(token=HF_TOKEN)
    REPO = "Exploration-Lab/CS-781-Capstone"
    FILES = {"train": "Phase_4_CJPE/Train/train.json",
             "validation": "Phase_4_CJPE/Train/dev.json",
             "test": "Phase_4_CJPE/Test/test.json"}
    DATA_DIR = Path("/content/cjpe_data"); DATA_DIR.mkdir(exist_ok=True)

    for split in SPLITS:
        if cache_path(split).exists():
            continue
        print(f"\n[{split}] downloading and loading")
        local = hf_hub_download(repo_id=REPO, filename=FILES[split], repo_type="dataset",
                                token=HF_TOKEN, local_dir=str(DATA_DIR))
        raw = load_json_records(local)
        seen, ids, texts, labels = set(), [], [], []
        for r in raw:
            cid = str(r["id"])
            if cid in seen:
                continue
            seen.add(cid); ids.append(cid); texts.append(r.get("text") or "")
            if split != "test":                      # test labels are never read
                labels.append(int(r["label"]))
        print(f"   rows {len(raw):,} -> unique cases {len(ids):,}")
        del raw; gc.collect()
        res = tokenize_texts(texts)
        del texts; gc.collect()

        off = np.zeros(len(res) + 1, np.int64)
        off[1:] = np.cumsum([len(r[0]) for r in res])
        payload = dict(ids=np.array(ids), tail_flat=np.concatenate([r[0] for r in res]), tail_off=off,
                       full=np.array([r[1] for r in res], bool),
                       nchars=np.array([r[2] for r in res], np.int64),
                       preview=np.array([r[3] for r in res]),
                       orig_no_upper=np.array([r[4] for r in res], bool))
        if split != "test":
            payload["labels"] = np.array(labels, np.int64)
        np.savez(cache_path(split), **payload)
        print("   saved", cache_path(split).name)
        del res, payload; gc.collect()
        os.remove(local)
print("token caches ready.")

v2 token caches on Drive: True | /content/drive/MyDrive/phase4_exp1_v2_encoder_full_ft/token_cache
token caches ready.


In [5]:
D = {}
for s in SPLITS:
    z = np.load(cache_path(s), allow_pickle=False)
    D[s] = {k: z[k] for k in z.files}
    D[s]["n"] = len(D[s]["ids"])
    assert len(set(D[s]["ids"].tolist())) == D[s]["n"]

def get_tail(s, i): d = D[s]; return d["tail_flat"][d["tail_off"][i]:d["tail_off"][i + 1]]
def label_of(s, i): return int(D[s]["labels"][i]) if "labels" in D[s] else -1
def n_windows(s, i):
    L = int(D[s]["tail_off"][i + 1] - D[s]["tail_off"][i])
    return min(K, max(1, -(-L // BODY)))

print(f"{'split':<12}{'cases':>8}{'pos rate':>10}{'whole doc kept':>16}{'avg windows':>13}{'orig. lower-case':>18}")
for s in SPLITS:
    d = D[s]
    pos = f"{d['labels'].mean():.3f}" if "labels" in d else "n/a"
    nw = np.array([n_windows(s, i) for i in range(d["n"])])
    print(f"{s:<12}{d['n']:>8,}{pos:>10}{100*d['full'].mean():>15.1f}%{nw.mean():>13.2f}{100*d['orig_no_upper'].mean():>17.1f}%")
assert max(int(D[s]["tail_flat"].max()) for s in SPLITS) < 50000
assert D["test"]["n"] == 1517 and "labels" not in D["test"]
assert all((get_tail(s, 0) != SPACE_ID).all() for s in SPLITS)
print("\nsample tail (decoded):", TOK.decode(get_tail("train", 0)[:40].tolist())[:160])

split          cases  pos rate  whole doc kept  avg windows  orig. lower-case
train         32,305     0.414           73.9%         2.68              0.0%
validation       994     0.500           64.3%         3.23            100.0%
test           1,517       n/a           62.6%         3.26             98.9%

sample tail (decoded): atholiccollege6wascompanycernedwithselectionandappointmentofaprincipalinanunaidedminorityeducationalinstitution.itwasstatedinpara19thattherightcompanyferredonmi


## 5. Model: Phase-1 SWA decoder + document head

`ModelConfig`, RoPE, `RMSNorm`, `SwiGLU`, `CausalSelfAttention` (chunked local sliding-window + global GQA) and `TransformerBlock` are the **Phase-1 classes verbatim** — the sliding-window attention is the thing under test, so it is not replaced by SDPA. `LegalDecoder` is `LegalDecoderLM` minus the tied `lm_head`, so the parameter names still match the checkpoint. Residual dropout is added exactly as in the GQA-decoder run (a parameterless `nn.Dropout`, so checkpoint loading is unaffected). The head is identical to the GQA-decoder run: last-token + mean pooling → LayerNorm → attention over chunks.

In [6]:
@dataclass
class ModelConfig:
    vocab_size: int = 50_000
    d_model: int = 768
    n_layers: int = 16
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_seq_len: int = 1024
    local_window: int = 128
    local_chunk_size: int = 256
    global_layer_indices: tuple = (3, 7, 11, 15)        # layers 4, 8, 12, 16
    rope_base: float = 10_000.0
    rms_norm_eps: float = 1e-6
    dropout: float = 0.0

def build_rope_cache(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()

def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    cos = cos.to(device=x.device, dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    sin = sin.to(device=x.device, dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    return x * cos + rotate_half(x) * sin

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class CausalSelfAttention(nn.Module):
    # Phase-1 scratch GQA + RoPE + local/global sliding-window attention (verbatim).
    # Right-padding never leaks into earlier tokens under causal masking, so no key-padding mask.
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, rope_base=10000.0,
                 dropout=0.0, is_global=True, local_window=128, local_chunk_size=256):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        assert max_seq_len > 0 and local_window > 0 and local_chunk_size > 0
        self.d_model, self.n_heads, self.n_kv_heads = d_model, n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.max_seq_len, self.dropout, self.is_global = max_seq_len, dropout, is_global
        self.local_window, self.local_chunk_size = local_window, local_chunk_size
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        cos, sin = build_rope_cache(max_seq_len, self.head_dim, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _split_heads(self, x, n_heads):
        B, T, _ = x.shape
        return x.reshape(B, T, n_heads, self.head_dim).transpose(1, 2)

    def _apply_dropout(self, w):
        if self.dropout > 0.0 and self.training:
            w = F.dropout(w, p=self.dropout)
        return w

    def _global_attention(self, q, k, v, past_len):
        B, Hq, T, Dh = q.shape
        total_len = k.size(2)
        q_grouped = q.reshape(B, self.n_kv_heads, self.n_rep, T, Dh)
        k_grouped, v_grouped = k.unsqueeze(2), v.unsqueeze(2)
        scores = torch.matmul(q_grouped, k_grouped.transpose(-2, -1)) / math.sqrt(Dh)
        causal_mask = torch.ones(T, total_len, dtype=torch.bool, device=q.device).triu(diagonal=1 + past_len)
        scores = scores.masked_fill(causal_mask[None, None, None, :, :], torch.finfo(scores.dtype).min)
        w = self._apply_dropout(F.softmax(scores, dim=-1))
        return torch.matmul(w, v_grouped).reshape(B, self.n_heads, T, Dh)

    def _local_attention_chunked(self, q, k, v, past_len):
        B, Hq, T, Dh = q.shape
        outputs, chunk_size = [], self.local_chunk_size
        for q_start in range(0, T, chunk_size):
            q_end = min(q_start + chunk_size, T)
            q_chunk = q[:, :, q_start:q_end, :]
            abs_q_start, abs_q_end = past_len + q_start, past_len + q_end
            k_start = max(0, abs_q_start - self.local_window + 1)
            k_end = abs_q_end
            k_range, v_range = k[:, :, k_start:k_end, :], v[:, :, k_start:k_end, :]
            q_grouped = q_chunk.reshape(B, self.n_kv_heads, self.n_rep, q_end - q_start, Dh)
            k_grouped, v_grouped = k_range.unsqueeze(2), v_range.unsqueeze(2)
            scores = torch.matmul(q_grouped, k_grouped.transpose(-2, -1)) / math.sqrt(Dh)
            qp = torch.arange(abs_q_start, abs_q_end, device=q.device)
            kp = torch.arange(k_start, k_end, device=q.device)
            valid = (kp[None, :] <= qp[:, None]) & (kp[None, :] >= (qp[:, None] - self.local_window + 1))
            scores = scores.masked_fill(~valid[None, None, None, :, :], torch.finfo(scores.dtype).min)
            w = self._apply_dropout(F.softmax(scores, dim=-1))
            outputs.append(torch.matmul(w, v_grouped).reshape(B, self.n_heads, q_end - q_start, Dh))
        return torch.cat(outputs, dim=2)

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, _ = x.shape
        past_len = 0 if past_kv is None else past_kv[0].size(2)
        if past_len + T > self.max_seq_len:
            raise ValueError("sequence length exceeds max_seq_len")
        q = self._split_heads(self.q_proj(x), self.n_heads)
        k = self._split_heads(self.k_proj(x), self.n_kv_heads)
        v = self._split_heads(self.v_proj(x), self.n_kv_heads)
        cos, sin = self.rope_cos[past_len:past_len + T], self.rope_sin[past_len:past_len + T]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2); v = torch.cat([past_kv[1], v], dim=2)
        present_kv = (k, v) if use_cache else None
        out = self._global_attention(q, k, v, past_len) if self.is_global \
              else self._local_attention_chunked(q, k, v, past_len)
        out = out.transpose(1, 2).contiguous().reshape(B, T, self.d_model)
        return self.out_proj(out), present_kv

class TransformerBlock(nn.Module):
    def __init__(self, c, is_global, p_drop=0.0):
        super().__init__()
        self.is_global = is_global
        self.norm1 = RMSNorm(c.d_model, c.rms_norm_eps)
        self.attn = CausalSelfAttention(d_model=c.d_model, n_heads=c.n_heads, n_kv_heads=c.n_kv_heads,
                                        max_seq_len=c.max_seq_len, rope_base=c.rope_base, dropout=c.dropout,
                                        is_global=is_global, local_window=c.local_window,
                                        local_chunk_size=c.local_chunk_size)
        self.norm2 = RMSNorm(c.d_model, c.rms_norm_eps)
        self.mlp = SwiGLU(c.d_model, c.d_ff)
        self.drop = nn.Dropout(p_drop)          # no parameters -> checkpoint-compatible
    def forward(self, x):
        attn_out, _ = self.attn(self.norm1(x), past_kv=None, use_cache=False)
        x = x + self.drop(attn_out)
        return x + self.drop(self.mlp(self.norm2(x)))

class LegalDecoder(nn.Module):
    # parameter names match the Phase-1 LegalDecoderLM (the tied lm_head is not needed here)
    def __init__(self, c, grad_ckpt=False, p_drop=0.0):
        super().__init__()
        self.cfg, self.grad_ckpt = c, grad_ckpt
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(c, i in tuple(c.global_layer_indices), p_drop)
                                     for i in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)
    def forward(self, ids):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("input sequence is longer than max_seq_len")
        x = self.token_embedding(ids)
        for blk in self.blocks:
            x = checkpoint(blk, x, use_reentrant=False) if (self.grad_ckpt and self.training) else blk(x)
        return self.final_norm(x)

class LegalDocClassifier(nn.Module):
    # window vector = [last-token state, mean of all states] -> LayerNorm; document = attention over windows
    # (+ distance-from-end embedding) concatenated with the last window's vector; linear classifier.
    def __init__(self, c, max_chunks=K, resid_drop=0.0, head_drop=0.1, grad_ckpt=False):
        super().__init__()
        d = c.d_model; dv = 2 * d
        self.max_chunks = max_chunks
        self.backbone = LegalDecoder(c, grad_ckpt, resid_drop)
        self.pool_norm = nn.LayerNorm(dv)
        self.chunk_pos = nn.Embedding(max_chunks, dv)
        nn.init.normal_(self.chunk_pos.weight, std=0.02)
        self.chunk_attn = nn.Sequential(nn.Linear(dv, 256), nn.Tanh(), nn.Linear(256, 1, bias=False))
        self.dropout = nn.Dropout(head_drop)
        self.classifier = nn.Linear(2 * dv, 2)
        nn.init.normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)
        self.last_chunk_attn, self.last_chunk_mask = None, None

    def forward(self, batch):
        am = batch["attn_mask"]
        h = self.backbone(batch["input_ids"]).float()                          # [N, L, d]
        lens = am.sum(1)
        last = h[torch.arange(h.size(0), device=h.device), lens - 1]           # state at the closing </s>
        m = am.unsqueeze(-1).float()
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        v = self.pool_norm(torch.cat([last, mean], dim=-1))                    # [N, 2d]
        B, Kc, dv = batch["n_docs"], self.max_chunks, v.shape[-1]
        di, sl = batch["doc_index"], batch["slot"]
        chunks = v.new_zeros(B, Kc, dv).index_put((di, sl), v)
        cm = torch.zeros(B, Kc, dtype=torch.bool, device=v.device)
        cm[di, sl] = True
        pe = self.chunk_pos(torch.arange(Kc - 1, -1, -1, device=v.device))     # slot s -> distance Kc-1-s from the end
        sc = self.chunk_attn(chunks + pe.unsqueeze(0)).squeeze(-1).float().masked_fill(~cm, float("-inf"))
        w = torch.softmax(sc, dim=-1)
        agg = (w.unsqueeze(-1) * chunks).sum(1)
        doc = torch.cat([agg, chunks[:, Kc - 1]], dim=-1)                      # last chunk always present
        self.last_chunk_attn, self.last_chunk_mask = w.detach(), cm
        return self.classifier(self.dropout(doc))

# ---- weights-only copy of the Phase-1 checkpoint (the original also holds optimiser + RNG state) ----
SLIM = OUT_DIR / "phase1_swa_decoder_weights.pt"
if not SLIM.exists():
    _ck = torch.load(CKPT_DRIVE, map_location="cpu", weights_only=False, mmap=True)   # your own Phase-1 checkpoint
    torch.save(dict(model_state_dict=_ck["model_state_dict"], model_config=_ck["model_config"],
                    global_step=int(_ck.get("global_step", -1)),
                    best_val_loss=float(_ck.get("best_val_loss", float("nan")))), SLIM)
    del _ck; gc.collect()
P1 = torch.load(SLIM, map_location="cpu", weights_only=True)
_p1_cfg = dict(P1["model_config"])
_p1_cfg["global_layer_indices"] = tuple(_p1_cfg["global_layer_indices"])
P1_CFG = ModelConfig(**_p1_cfg)
assert P1_CFG.max_seq_len == WIN, f"checkpoint context {P1_CFG.max_seq_len} != WIN {WIN}"
print(f"Phase-1 SWA decoder: step {P1['global_step']} | best validation LM loss {P1['best_val_loss']:.4f}")
print("global (full-attention) layers:", [i + 1 for i in P1_CFG.global_layer_indices],
      "| local window", P1_CFG.local_window, "| context", P1_CFG.max_seq_len)

def load_phase1_weights(backbone):
    missing, unexpected = backbone.load_state_dict(P1["model_state_dict"], strict=False)
    assert not missing, f"missing keys: {missing}"
    assert set(unexpected) <= {"lm_head.weight"}, f"unexpected keys: {unexpected}"

Phase-1 SWA decoder: step 15000 | best validation LM loss 1.4405
global (full-attention) layers: [4, 8, 12, 16] | local window 128 | context 1024


## 6. LoRMA$_+$ from scratch — Assignment-4 Q4.4, adapted from GPT-2 to this model

`LORMA_VARIANT`, `_lorma_scaling`, `make_lorma_factors`, `lorma_dense` and `lorma_reordered` are the **Assignment functions unchanged**. Only the wrapper changes, because the layer type changes:

| Assignment (GPT-2) | here (Phase-1 SWA decoder) |
|---|---|
| `Conv1D`, weight `[d_in, d_out]`, `y = xW + b` | `nn.Linear`, weight `[d_out, d_in]`, `bias=False`, `y = xW^\top` |
| fused `c_attn` `[768, 2304]`, factors `2304 × r` | the **per-projection placement** from the Variants section: an independent factor set per projection, `A ∈ ℝ^{r×d_out}`, `B ∈ ℝ^{d_out×r}` |
| `targets ⊆ ("q","k","v")` selecting column blocks of one fused map | `targets ⊆ ("q_proj","k_proj","v_proj","out_proj")` selecting separate modules — GQA already keeps them separate, with `d_out = 768, 256, 256, 768` (12 query / 4 KV heads) |
| `h.{i}.attn.c_attn` | `blocks.{i}.attn.{q,k,v,out}_proj` |

The equation is untouched: $W'=\left(I+\tfrac{\alpha}{r}BA\right)W_0$, evaluated in the re-ordered form `lorma_reordered`, which multiplies the **output** side and never builds a $d_\text{out}\times d_\text{out}$ matrix. Because `d_out` differs per projection, the parameter count per block is $2r(768+256+256+768) = 2r\cdot 2048 = 32{,}768$ at $r=8$ — derived and checked against the counted value below.

Trainable: the LoRMA$_+$ factors plus the document head (`pool_norm`, `chunk_pos`, `chunk_attn`, `classifier`). The 16-layer backbone is frozen.

In [7]:
# ===========================================================================
#  LoRMA+ — Assignment-4 Q4.4 implementation.
#  `peft` and adapter libraries are NOT used anywhere in this notebook.
# ===========================================================================

LORMA_VARIANT = {"name": "LoRMA+",
                 "equation": "W' = (I + (alpha/r) B A) W_0",
                 "paper_version": "Bihany et al. (2025), Section 3.2.2, Additive Rank Inflation",
                 "alpha": LORMA_ALPHA}


def _lorma_scaling(rank, variant_config=None):
    config = LORMA_VARIANT if variant_config is None else variant_config
    return config.get("alpha", 16) / rank


def make_lorma_factors(d_in, d_out, rank, init=False, generator=None, dtype=torch.float64, swap_ab=False):
    if rank < 1:
        raise ValueError("rank must be positive")

    a = torch.randn(rank, d_out, generator=generator, dtype=dtype)
    b = torch.randn(d_out, rank, generator=generator, dtype=dtype)

    if init:
        if swap_ab:
            a = torch.zeros_like(a)
        else:
            b = torch.zeros_like(b)

    return a, b


def lorma_dense(x, frozen_weight, factors, variant_config=None):
    """Dense form: build (I + scale*BA) W_0 explicitly. frozen_weight is [d_out, d_in]."""
    a, b = factors
    scale = _lorma_scaling(a.size(0), variant_config)
    identity = torch.eye(
        frozen_weight.size(0),
        dtype=frozen_weight.dtype,
        device=frozen_weight.device
    )
    updated_weight = (identity + scale * (b @ a)) @ frozen_weight
    return x @ updated_weight.T


def lorma_reordered(x, frozen_weight, factors, variant_config=None):
    """Re-ordered form: never materialise the d_out x d_out product."""
    a, b = factors
    scale = _lorma_scaling(a.size(0), variant_config)
    base_output = x @ frozen_weight.T
    update = base_output @ a.T @ b.T
    return base_output + scale * update


class LoRMALinear(nn.Module):
    """LoRMA+ around a frozen nn.Linear (the nn.Linear analogue of the Assignment's LoRMAConv1D).

    The Assignment's Conv1D holds weight [d_in, d_out]; nn.Linear holds [d_out, d_in] and applies
    x @ W^T, so `lorma_reordered` is called with the Linear weight directly and the factors act on
    the d_out output rows, exactly as in the per-projection placement of Q4.4.
    """

    def __init__(self, base_linear, rank=8, variant_config=None, swap_ab=False):
        super().__init__()
        if rank < 1:
            raise ValueError("rank must be positive")
        assert isinstance(base_linear, nn.Linear)

        self.base = base_linear
        for param in self.base.parameters():
            param.requires_grad_(False)

        self.rank = rank
        self.variant_config = LORMA_VARIANT if variant_config is None else variant_config
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.merged = False

        device = self.base.weight.device
        dtype = self.base.weight.dtype
        a, b = make_lorma_factors(
            self.in_features, self.out_features, rank,
            init=True, dtype=dtype, swap_ab=swap_ab
        )
        self.lorma_A = nn.Parameter(a.to(device))
        self.lorma_B = nn.Parameter(b.to(device))

    def scaling(self):
        return _lorma_scaling(self.rank, self.variant_config)

    def forward(self, x):
        if self.merged:
            return self.base(x)
        output = lorma_reordered(
            x,
            self.base.weight,
            (self.lorma_A, self.lorma_B),
            self.variant_config
        )
        if self.base.bias is not None:
            output = output + self.base.bias
        return output

    @torch.no_grad()
    def merge(self):
        """Fold (I + scale*BA) into the frozen weight: zero-overhead inference."""
        if not self.merged:
            scale = self.scaling()
            identity = torch.eye(self.out_features, dtype=self.base.weight.dtype,
                                 device=self.base.weight.device)
            transform = identity + scale * (self.lorma_B @ self.lorma_A)
            self.base.weight.data.copy_(transform @ self.base.weight.data)
            self.merged = True

    def extra_repr(self):
        return (f"in={self.in_features}, out={self.out_features}, r={self.rank}, "
                f"alpha={self.variant_config.get('alpha')}")


def attach_lorma(model, rank=8, targets=LORMA_TARGETS, swap_ab=False, backbone_prefix="backbone."):
    """Wrap every targeted nn.Linear in LoRMALinear; train only those factors and the head.

    Returns (list of wrapped module paths, list of trainable parameter names).
    """
    wrapped = []

    def _walk(module, path):
        for name, child in list(module.named_children()):
            full = f"{path}.{name}" if path else name
            if name in targets and isinstance(child, nn.Linear):
                setattr(module, name, LoRMALinear(child, rank=rank, swap_ab=swap_ab))
                wrapped.append(full)
            else:
                _walk(child, full)

    _walk(model, "")

    for name, p in model.named_parameters():
        p.requires_grad_(("lorma_" in name) or (not name.startswith(backbone_prefix)))

    return wrapped, [n for n, p in model.named_parameters() if p.requires_grad]


def merge_lorma_weights(model):
    for m in model.modules():
        if isinstance(m, LoRMALinear):
            m.merge()
    return model


def lorma_param_report(model):
    total = sum(p.numel() for p in model.parameters())
    lorma = sum(p.numel() for n, p in model.named_parameters() if "lorma_" in n)
    head = sum(p.numel() for n, p in model.named_parameters()
               if p.requires_grad and "lorma_" not in n)
    return dict(total_params=total, lorma_params=lorma, head_params=head,
                trainable_params=lorma + head, trainable_fraction=(lorma + head) / total)


def build_model(pretrained=True):
    m = LegalDocClassifier(P1_CFG, K, RESID_DROPOUT, HEAD_DROPOUT, GRAD_CKPT)
    if pretrained:
        load_phase1_weights(m.backbone)      # load Phase-1 weights first, then wrap the projections
    attach_lorma(m, LORMA_R, LORMA_TARGETS, LORMA_SWAP_AB)
    return m.to(device)


_probe = build_model()
TRAINABLE_KEYS = [n for n, p in _probe.named_parameters() if p.requires_grad]
_rep = lorma_param_report(_probe)
_d_outs = {"q_proj": P1_CFG.d_model,
           "k_proj": P1_CFG.d_model // P1_CFG.n_heads * P1_CFG.n_kv_heads,
           "v_proj": P1_CFG.d_model // P1_CFG.n_heads * P1_CFG.n_kv_heads,
           "out_proj": P1_CFG.d_model}
_derived = P1_CFG.n_layers * sum(2 * LORMA_R * _d_outs[t] for t in LORMA_TARGETS)
print(f"LoRMA+ modules wrapped   : {sum(isinstance(mm, LoRMALinear) for mm in _probe.modules())}"
      f" ({len(LORMA_TARGETS)} per block x {P1_CFG.n_layers} blocks)")
print(f"factors per block        : " + ", ".join(f"{t} 2x{LORMA_R}x{_d_outs[t]}" for t in LORMA_TARGETS))
print(f"hand-derived LoRMA params: {_derived:,}")
print(f"counted LoRMA params     : {_rep['lorma_params']:,}")
assert _derived == _rep["lorma_params"], "hand-derived parameter count does not match the counted one"
print(f"total parameters         : {_rep['total_params']:,}")
print(f"  document head          : {_rep['head_params']:,}")
print(f"  trainable              : {_rep['trainable_params']:,} "
      f"({100*_rep['trainable_fraction']:.3f}% of the model)")
print(f"  frozen backbone        : {_rep['total_params'] - _rep['trainable_params']:,}")
del _probe; gc.collect(); torch.cuda.empty_cache()


def model_from_state(state):
    # frozen Phase-1 weights + the trained factors and head (the only keys in `state`)
    m = build_model(pretrained=True)
    missing, unexpected = m.load_state_dict(state, strict=False)
    assert not unexpected, f"unexpected keys: {unexpected}"
    assert set(state) == set(TRAINABLE_KEYS), "state does not match the trainable parameters"
    return m.eval()

LoRMA+ modules wrapped   : 64 (4 per block x 16 blocks)
factors per block        : q_proj 2x8x768, k_proj 2x8x256, v_proj 2x8x256, out_proj 2x8x768
hand-derived LoRMA params: 524,288
counted LoRMA params     : 524,288
total parameters         : 140,022,018
  document head          : 409,090
  trainable              : 933,378 (0.667% of the model)
  frozen backbone        : 139,088,640


### LoRMA$_+$ correctness checks (float64, CPU)

`check_lorma` is the Assignment's float64 check on a rectangular example: the **dense** and **re-ordered** forms must agree in outputs **and** gradients, the base-preserving initialisation must reproduce $xW_0^\top$ exactly, and SVD must show the rank bottleneck of the naive product against the rank-inflated transformation. Both initialisations (`swap_ab=False` → $B=0$; `swap_ab=True` → $A=0$) are checked.

In [8]:
def check_lorma(d_in=12, d_out=16, rank=3, seed=0):
    out = {}
    for swap_ab in (False, True):
        g = torch.Generator().manual_seed(seed)
        W0 = torch.randn(d_out, d_in, dtype=torch.float64, generator=g)     # [d_out, d_in]
        x = torch.randn(5, d_in, dtype=torch.float64, generator=g)

        # --- random factors: dense == re-ordered, in outputs and gradients ---
        a, b = make_lorma_factors(d_in, d_out, rank, init=False, generator=g,
                                  dtype=torch.float64, swap_ab=swap_ab)
        a1, b1 = a.clone().requires_grad_(True), b.clone().requires_grad_(True)
        a2, b2 = a.clone().requires_grad_(True), b.clone().requires_grad_(True)
        y_dense = lorma_dense(x, W0, (a1, b1))
        y_reord = lorma_reordered(x, W0, (a2, b2))
        d_out_err = float((y_dense - y_reord).abs().max())
        y_dense.pow(2).sum().backward()
        y_reord.pow(2).sum().backward()
        d_ga = float((a1.grad - a2.grad).abs().max())
        d_gb = float((b1.grad - b2.grad).abs().max())

        # --- base-preserving initialisation: exactly x @ W0.T ---
        ai, bi = make_lorma_factors(d_in, d_out, rank, init=True, generator=g,
                                   dtype=torch.float64, swap_ab=swap_ab)
        d_id = float((lorma_reordered(x, W0, (ai, bi)) - x @ W0.T).abs().max())

        # --- SVD ranks: naive multiplicative product vs the inflated transformation ---
        scale = _lorma_scaling(rank)
        naive = (b @ a) @ W0
        inflated = (torch.eye(d_out, dtype=torch.float64) + scale * (b @ a)) @ W0
        tol = 1e-8
        r_naive = int((torch.linalg.svdvals(naive) > tol).sum())
        r_infl = int((torch.linalg.svdvals(inflated) > tol).sum())
        r_fact = int((torch.linalg.svdvals(b @ a) > tol).sum())

        out[swap_ab] = dict(dense_vs_reordered=d_out_err, grad_A=d_ga, grad_B=d_gb,
                            identity_at_init=d_id, rank_factors=r_fact,
                            rank_naive_BA_W0=r_naive, rank_inflated=r_infl)
        assert d_out_err < 1e-10 and d_ga < 1e-10 and d_gb < 1e-10
        assert d_id == 0.0
        assert r_naive <= rank and r_infl == min(d_in, d_out)
    return out

for swap_ab, r in check_lorma().items():
    print(f"swap_ab={swap_ab} ({'A=0, B random' if swap_ab else 'B=0, A random'})")
    print(f"   dense vs re-ordered  : outputs {r['dense_vs_reordered']:.2e} | "
          f"grad A {r['grad_A']:.2e} | grad B {r['grad_B']:.2e}")
    print(f"   identity at init     : {r['identity_at_init']:.2e}  (exact)")
    print(f"   numerical ranks      : factors BA = {r['rank_factors']} | naive (BA)W0 = "
          f"{r['rank_naive_BA_W0']} | inflated (I+sBA)W0 = {r['rank_inflated']}")
print("\nLoRMA+ checks passed: the re-ordered form matches the dense form, the initialisation preserves")
print("the base map exactly, and additive rank inflation lifts the rank-3 bottleneck to full rank.")

# the wrapped module must reproduce the frozen projection exactly at step 0
_lin = nn.Linear(16, 24, bias=False)
_w = LoRMALinear(copy.deepcopy(_lin), rank=LORMA_R, swap_ab=LORMA_SWAP_AB)
_x = torch.randn(3, 16)
print("LoRMALinear identity at init: max |delta| =",
      float((_w(_x) - _lin(_x)).abs().max()))
assert float((_w(_x) - _lin(_x)).abs().max()) < 1e-6
del _lin, _w, _x

swap_ab=False (B=0, A random)
   dense vs re-ordered  : outputs 8.53e-14 | grad A 4.37e-11 | grad B 2.91e-11
   identity at init     : 0.00e+00  (exact)
   numerical ranks      : factors BA = 3 | naive (BA)W0 = 3 | inflated (I+sBA)W0 = 12
swap_ab=True (A=0, B random)
   dense vs re-ordered  : outputs 8.53e-14 | grad A 4.37e-11 | grad B 2.91e-11
   identity at init     : 0.00e+00  (exact)
   numerical ranks      : factors BA = 3 | naive (BA)W0 = 3 | inflated (I+sBA)W0 = 12

LoRMA+ checks passed: the re-ordered form matches the dense form, the initialisation preserves
the base map exactly, and additive rank inflation lifts the rank-3 bottleneck to full rank.
LoRMALinear identity at init: max |delta| = 0.0


/tmp/ipykernel_5115/3865852304.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  d_out_err = float((y_dense - y_reord).abs().max())


## 7. Document → windows, micro-batch packing

Windows are the **last K = 4 chunks** of the compact, lower-cased document, each `</s>` + 1022 tokens + `</s>` = 1024 (oldest chunk possibly shorter) — the same ~4,08x compact tokens the GQA run covered with 8 × 510. `<s>` is never used, because the Phase-1 decoder was trained with `</s>` as the document boundary; the closing `</s>` gives the last-token pooling a natural summary position. Each non-final chunk is dropped with probability 0.1 during training. Micro-batches hold at most 8 windows of 1,024 tokens, with gradient checkpointing on.

In [9]:
def build_chunks(split, i, drop_p=0.0, rng=None):
    tail = get_tail(split, i).astype(np.int64)
    body, end = [], len(tail)
    while end > 0 and len(body) < K:
        start = max(0, end - BODY)
        body.append(tail[start:end]); end = start
    body = body[::-1] if body else [tail]
    n = len(body)
    keep = [True] * n
    if drop_p > 0 and n > 1:
        for j in range(n - 1):                         # never drop the final chunk
            if rng.random() < drop_p:
                keep[j] = False
    return [(K - n + j, np.concatenate([[EOS_ID], c, [EOS_ID]]).astype(np.int64)) for j, c in enumerate(body) if keep[j]]

def collate(items, train=False, rng=None):
    seqs, didx, slot = [], [], []
    for b, (s, i) in enumerate(items):
        for sl, c in build_chunks(s, i, CHUNK_DROP if train else 0.0, rng):
            seqs.append(c); didx.append(b); slot.append(sl)
    L = max(len(c) for c in seqs)
    ids = np.zeros((len(seqs), L), np.int64); am = np.zeros((len(seqs), L), bool)
    for r, c in enumerate(seqs):
        ids[r, :len(c)] = c; am[r, :len(c)] = True
    return dict(input_ids=torch.from_numpy(ids), attn_mask=torch.from_numpy(am),
                doc_index=torch.tensor(didx, dtype=torch.long), slot=torch.tensor(slot, dtype=torch.long),
                n_docs=len(items))

def pack(items, max_windows):
    groups, cur, w = [], [], 0
    for it in items:
        n = n_windows(*it)
        if cur and w + n > max_windows:
            groups.append(cur); cur, w = [], 0
        cur.append(it); w += n
    if cur:
        groups.append(cur)
    return groups

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}

_b = collate([("train", i) for i in range(4)])
assert _b["input_ids"].shape[1] <= WIN
print("packing check | windows", tuple(_b["input_ids"].shape), "| slots", _b["slot"].tolist())

# sanity: Phase-1 weights + lower-cased compact tokens are coherent (next-token loss far below ln(50000) = 10.82)
@torch.inference_mode()
def lm_sanity(n_docs=4):
    bb = LegalDecoder(P1_CFG).to(device).eval(); load_phase1_weights(bb)
    seqs = [build_chunks("train", i_)[-1][1] for i_ in range(n_docs)]
    L = min(len(s_) for s_ in seqs)
    ids = torch.from_numpy(np.stack([s_[:L] for s_ in seqs])).to(device)
    with torch.autocast("cuda", dtype=AMP_DTYPE):
        logits = F.linear(bb(ids)[:, :-1], bb.token_embedding.weight)
    loss = float(F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1)))
    del bb; gc.collect(); torch.cuda.empty_cache()
    return loss
print(f"Phase-1 next-token loss on lower-cased compact CJPE windows: {lm_sanity():.3f}   "
      f"(its own validation loss {P1['best_val_loss']:.2f}; random guessing {math.log(50000):.2f})")

packing check | windows (16, 1024) | slots [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]
Phase-1 next-token loss on lower-cased compact CJPE windows: 6.612   (its own validation loss 1.44; random guessing 10.82)


## 8. Training utilities (EMA over the trainable weights, half-epoch evaluation, memory tracking)

In [10]:
def tune_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    f = np.array([f1_score(y, (p >= t).astype(int), average="macro") for t in grid])
    sm = np.convolve(np.pad(f, 2, mode="edge"), np.ones(5) / 5, mode="valid")
    k = int(np.argmax(sm)); return float(grid[k]), float(f[k])

def prior_threshold(p, rate):
    # threshold that labels the top `rate` fraction of documents as class 1
    return float(np.quantile(p, 1.0 - rate))

def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    return dict(macro_f1=float(f1_score(y, pred, average="macro")),
                precision_macro=float(precision_score(y, pred, average="macro", zero_division=0)),
                recall_macro=float(recall_score(y, pred, average="macro", zero_division=0)),
                f1_per_class=[float(x) for x in f1_score(y, pred, average=None)],
                recall_per_class=[float(x) for x in recall_score(y, pred, average=None, zero_division=0)],
                pred_pos_rate=float(pred.mean()))

@torch.inference_mode()
def predict_proba(model, items):
    was = model.training; model.eval()
    nw = np.array([n_windows(s, i) for s, i in items])
    order = np.argsort(-nw, kind="stable")
    ordered = [items[j] for j in order]
    out = np.zeros(len(items)); asum, acnt = np.zeros(K), np.zeros(K); pos = 0
    for g in pack(ordered, EVAL_WINDOWS):
        batch = to_dev(collate(g))
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(batch)
        out[order[pos:pos + len(g)]] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        pos += len(g)
        w, cm = model.last_chunk_attn.cpu().numpy(), model.last_chunk_mask.cpu().numpy()
        asum += (w * cm).sum(0); acnt += cm.sum(0)
    model.train(was)
    return out, (asum / np.maximum(acnt, 1)).tolist()

class EMA:
    # exponential moving average of the trainable weights only (the backbone is frozen,
    # so an EMA over it would be the identity anyway)
    def __init__(self, model, decay):
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.keys = [n for n, p in model.named_parameters() if p.requires_grad]
        src = dict(model.named_parameters()); dst = dict(self.model.named_parameters())
        self.src = [src[k] for k in self.keys]
        self.dst = [dst[k] for k in self.keys]
        self.decay, self.n = decay, 0
    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_mul_(self.dst, d)
        torch._foreach_add_(self.dst, [p.detach() for p in self.src], alpha=1 - d)
    def state(self):
        sd = self.model.state_dict()
        return {k: sd[k].detach().cpu().clone() for k in self.keys}

def build_optimizer(model):
    # layer-wise LR decay over the LoRMA factors (deepest block gets the full LR); head on its own LR
    L = P1_CFG.n_layers
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone."):                 # LoRMA+ factors
            depth = L - 1 - int(name.split(".")[2]) if name.startswith("backbone.blocks.") else 0
            lr, wd = LORMA_LR * (LLRD ** depth), LORMA_WD
        else:                                            # document head
            lr = HEAD_LR
            wd = 0.0 if p.ndim == 1 else WD
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([dict(params=ps, lr=lr, weight_decay=wd) for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)

def train_model(train_items, eval_items=None, label=""):
    # one run of E_MAX epochs; the LR schedule spans the whole run
    set_seed(SEED); torch.cuda.reset_peak_memory_stats()
    model = build_model(); model.train()
    ema = EMA(model, EMA_DECAY)
    y_tr = np.array([label_of(s, i) for s, i in train_items])
    n0, n1 = int((y_tr == 0).sum()), int((y_tr == 1).sum())
    cw = torch.tensor([len(y_tr) / (2 * n0), len(y_tr) / (2 * n1)], dtype=torch.float32, device=device)
    steps_per_epoch = DPE // EFF_DOCS
    total = E_MAX * steps_per_epoch
    run_steps = total
    warm = max(1, int(WARMUP * total))
    opt = build_optimizer(model)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    rng = np.random.default_rng(SEED); crng = np.random.default_rng(SEED + 1)
    order, ptr = rng.permutation(len(train_items)), 0
    y_ev = np.array([label_of(s, i) for s, i in eval_items]) if eval_items else None
    rate = TARGET_POS_RATE if TARGET_POS_RATE is not None else (float(y_ev.mean()) if y_ev is not None else 0.5)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"[{label}] documents {len(train_items):,} | steps {run_steps} of {total} | "
          f"trainable params {trainable:,} | class weights {[round(x,3) for x in cw.tolist()]}")
    history, best, best_probs, best_attn, best_state = [], -1.0, None, None, None
    final_probs, final_attn, final_auc = None, None, None
    best_step = run_steps
    loss_acc, n_acc = 0.0, 0
    for step in range(1, run_steps + 1):
        take = []
        for _ in range(EFF_DOCS):
            if ptr >= len(order):
                order, ptr = rng.permutation(len(train_items)), 0
            take.append(train_items[int(order[ptr])]); ptr += 1
        opt.zero_grad(set_to_none=True)
        for g in pack(take, MICRO_WINDOWS):
            batch = to_dev(collate(g, train=True, rng=crng))
            yb = torch.tensor([label_of(s, i) for s, i in g], device=device)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(batch)
            loss = F.cross_entropy(logits.float(), yb, weight=cw, reduction="sum",
                                   label_smoothing=LABEL_SMOOTH) / EFF_DOCS
            scaler.scale(loss).backward()
            loss_acc += float(loss.detach())
        n_acc += 1
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], CLIP)
        scaler.step(opt); scaler.update(); sched.step()
        ema.update()

        if eval_items is not None and (step % EVAL_EVERY == 0 or step == run_steps):
            p, attn = predict_proba(ema.model, eval_items)
            thr, f_thr = tune_threshold(y_ev, p)
            rec = dict(step=step, epoch=step / steps_per_epoch, train_loss=loss_acc / n_acc,
                       auc=float(roc_auc_score(y_ev, p)),
                       f1_at_0p5=float(f1_score(y_ev, (p >= 0.5).astype(int), average="macro")),
                       tuned_threshold=thr, f1_at_tuned=f_thr,
                       f1_at_prior=float(f1_score(y_ev, (p >= prior_threshold(p, rate)).astype(int), average="macro")))
            history.append(rec)
            print(f"[{label}] step {step:>5} (epoch {rec['epoch']:.1f}) | train loss {rec['train_loss']:.4f} | dev AUC {rec['auc']:.4f} | "
                  f"F1@0.5 {rec['f1_at_0p5']:.4f} | F1@tuned({thr:.2f}) {f_thr:.4f} | F1@top-{rate:.0%} {rec['f1_at_prior']:.4f}")
            loss_acc, n_acc = 0.0, 0
            if step == run_steps:                      # dev predictions of the final (epoch-4) EMA weights
                final_probs, final_attn, final_auc = p.copy(), attn, rec["auc"]
            if rec["auc"] > best:
                best, best_probs, best_attn = rec["auc"], p.copy(), attn
                best_state = ema.state(); best_step = step
    out = dict(history=history, peak_mem_gb=torch.cuda.max_memory_allocated() / 1024**3,
               best_step=best_step, best_auc=best,
               best_probs=best_probs, best_chunk_attn=best_attn, best_state=best_state,
               final_probs=final_probs, final_chunk_attn=final_attn, final_auc=final_auc,
               final_state=ema.state())
    del model, ema, opt, sched; gc.collect(); torch.cuda.empty_cache()
    return out

train_items_all = [("train", i) for i in range(D["train"]["n"])]
dev_items = [("validation", i) for i in range(D["validation"]["n"])]
test_items = [("test", i) for i in range(D["test"]["n"])]
y_dev = np.array([label_of(s, i) for s, i in dev_items])
print("ready | train", len(train_items_all), "| dev", len(dev_items), "| test", len(test_items))

ready | train 32305 | dev 994 | test 1517


## 9. Training run — LoRMA$_+$ on the lower-cased training set, 4 epochs, dev every half epoch

One run, 4 epochs (4,000 optimiser steps). Only the factors and the head move; the Phase-1 SWA weights never change. Dev AUC / F1 of the EMA weights are logged every half epoch for the report. The submission uses the weights selected by `SUBMIT_WEIGHTS` (default: the final epoch-4 weights). Saved states are factors + head only, a few MB each.

In [11]:
RUN_JSON  = OUT_DIR / "run_summary.json"
RUN_PROBS = OUT_DIR / "run_dev_probs.npz"        # dev probabilities at the best step and at the final step
RUN_STATE = OUT_DIR / "run_lorma_states.pt"      # LoRMA+ factors + head: {"best": ..., "final": ...}
if RUN_JSON.exists() and RUN_PROBS.exists() and RUN_STATE.exists():
    R = json.load(open(RUN_JSON))
    z = np.load(RUN_PROBS); R["best_probs"], R["final_probs"] = z["best"], z["final"]
    st = torch.load(RUN_STATE, map_location="cpu"); R["best_state"], R["final_state"] = st["best"], st["final"]
    print("Training run loaded from Drive (delete run_* files to re-run).")
else:
    R = train_model(train_items_all, eval_items=dev_items, label="train")
    np.savez(RUN_PROBS, best=R["best_probs"], final=R["final_probs"])
    torch.save(dict(best=R["best_state"], final=R["final_state"]), RUN_STATE)
    json.dump({k: v for k, v in R.items() if k not in ("best_probs", "final_probs", "best_state", "final_state")},
              open(RUN_JSON, "w"), indent=1)

steps_per_epoch = DPE // EFF_DOCS
if SUBMIT_WEIGHTS == "final":
    sub_state, dev_probs, sub_attn = R["final_state"], R["final_probs"], R["final_chunk_attn"]
    sub_step, sub_auc = E_MAX * steps_per_epoch, R["final_auc"]
else:
    sub_state, dev_probs, sub_attn = R["best_state"], R["best_probs"], R["best_chunk_attn"]
    sub_step, sub_auc = int(R["best_step"]), R["best_auc"]

print(f"\nfinal weights (epoch {E_MAX}, step {E_MAX * steps_per_epoch}): dev AUC {R['final_auc']:.4f}")
print(f"best evaluation (step {int(R['best_step'])}, epoch {R['best_step'] / steps_per_epoch:.1f}): dev AUC {R['best_auc']:.4f}")
print(f"-> submission will use the {SUBMIT_WEIGHTS.upper()} weights (step {sub_step}, dev AUC {sub_auc:.4f})")
print(f"peak GPU memory {R['peak_mem_gb']:.1f} GB")
print("factors + head checkpoint size:", f"{RUN_STATE.stat().st_size/1024**2:.1f} MB (best + final)")


[train] documents 32,305 | steps 2000 of 2000 | trainable params 933,378 | class weights [0.854, 1.207]
[train] step   250 (epoch 0.5) | train loss 0.7881 | dev AUC 0.6018 | F1@0.5 0.5544 | F1@tuned(0.51) 0.5744 | F1@top-50% 0.5662
[train] step   500 (epoch 1.0) | train loss 0.7256 | dev AUC 0.6182 | F1@0.5 0.5688 | F1@tuned(0.48) 0.5762 | F1@top-50% 0.5674
[train] step   750 (epoch 1.5) | train loss 0.6907 | dev AUC 0.6512 | F1@0.5 0.5961 | F1@tuned(0.48) 0.6055 | F1@top-50% 0.6026
[train] step  1000 (epoch 2.0) | train loss 0.6656 | dev AUC 0.6755 | F1@0.5 0.6086 | F1@tuned(0.46) 0.6278 | F1@top-50% 0.6147
[train] step  1250 (epoch 2.5) | train loss 0.6443 | dev AUC 0.6748 | F1@0.5 0.6026 | F1@tuned(0.45) 0.6327 | F1@top-50% 0.6268
[train] step  1500 (epoch 3.0) | train loss 0.6342 | dev AUC 0.6894 | F1@0.5 0.6191 | F1@tuned(0.45) 0.6272 | F1@top-50% 0.6318
[train] step  1750 (epoch 3.5) | train loss 0.6231 | dev AUC 0.6979 | F1@0.5 0.6286 | F1@tuned(0.51) 0.6301 | F1@top-50% 0.6328


## 10. Dev analysis (for the report)

In [12]:
y, p = y_dev, dev_probs
print(f"dev analysis of the {SUBMIT_WEIGHTS} weights (step {sub_step}, epoch {sub_step / steps_per_epoch:.1f})\n")
rate = TARGET_POS_RATE if TARGET_POS_RATE is not None else float(y.mean())
thr_dev, _ = tune_threshold(y, p)
thr_prior = prior_threshold(p, rate)
rows = [("threshold 0.50", 0.5), (f"dev-tuned ({thr_dev:.2f})", thr_dev), (f"top-{rate:.0%} ({thr_prior:.2f})", thr_prior)]
print(f"{'':<24}{'macro-F1':>10}{'macro-P':>10}{'macro-R':>10}{'F1 cls0':>10}{'F1 cls1':>10}{'R cls0':>9}{'R cls1':>9}{'pred+':>8}")
for name, t in rows:
    m = metrics_at(y, p, t)
    print(f"{name:<24}{m['macro_f1']:>10.4f}{m['precision_macro']:>10.4f}{m['recall_macro']:>10.4f}"
          f"{m['f1_per_class'][0]:>10.4f}{m['f1_per_class'][1]:>10.4f}{m['recall_per_class'][0]:>9.4f}{m['recall_per_class'][1]:>9.4f}{m['pred_pos_rate']:>8.3f}")
print(f"ROC-AUC {roc_auc_score(y, p):.4f}")

pred = (p >= thr_prior).astype(int); d = D["validation"]
print("\nconfusion matrix (rows = truth, cols = prediction):\n", confusion_matrix(y, pred))
fp, fn, ok = (pred == 1) & (y == 0), (pred == 0) & (y == 1), pred == y
print(f"\n{'group':<16}{'count':>7}{'median chars':>14}{'mean P(1)':>11}")
for name, mk in [("correct", ok), ("false positive", fp), ("false negative", fn)]:
    print(f"{name:<16}{int(mk.sum()):>7}{np.median(d['nchars'][mk]):>14,.0f}{p[mk].mean():>11.3f}")
q = np.quantile(d["nchars"], [0, .25, .5, .75, 1.0])
print("\nerror rate by length quartile:")
for lo, hi in zip(q[:-1], q[1:]):
    mk = (d["nchars"] >= lo) & (d["nchars"] <= hi)
    print(f"  {lo:>9,.0f} - {hi:>9,.0f} chars: {100*(1-ok[mk].mean()):5.1f}%  (n={int(mk.sum())})")
nw = np.array([n_windows("validation", i) for i in range(d["n"])])
print("\nerror rate by number of windows used:")
for w_ in range(1, K + 1):
    mk = nw == w_
    if mk.any(): print(f"  {w_} window(s): {100*(1-ok[mk].mean()):5.1f}%  (n={int(mk.sum())})")
print("\nmean document-attention per chunk (oldest -> last):", [round(x, 3) for x in sub_attn])
print("\nlearning curve (EMA weights):")
for r in R["history"]:
    print(f"  step {r['step']:>5}  AUC {r['auc']:.4f}  F1@top-{rate:.0%} {r['f1_at_prior']:.4f}")

dev analysis of the final weights (step 2000, epoch 4.0)

                          macro-F1   macro-P   macro-R   F1 cls0   F1 cls1   R cls0   R cls1   pred+
threshold 0.50              0.6255    0.6441    0.6328    0.6778    0.5731   0.7726   0.4930   0.360
dev-tuned (0.45)            0.6361    0.6379    0.6368    0.6519    0.6204   0.6801   0.5936   0.457
top-50% (0.42)              0.6318    0.6318    0.6318    0.6303    0.6333   0.6278   0.6358   0.504
ROC-AUC 0.7005

confusion matrix (rows = truth, cols = prediction):
 [[312 185]
 [181 316]]

group             count  median chars  mean P(1)
correct             628        16,660      0.452
false positive      185        17,681      0.542
false negative      181        15,001      0.307

error rate by length quartile:
        894 -    10,649 chars:  36.5%  (n=249)
     10,649 -    16,656 chars:  37.5%  (n=248)
     16,656 -    26,466 chars:  39.1%  (n=248)
     26,466 -   259,571 chars:  34.1%  (n=249)

error rate by number of wind

## 11. Test predictions, submission file, model file

In [13]:
mF = model_from_state(sub_state); p_test, _ = predict_proba(mF, test_items)
np.save(OUT_DIR / "test_probs.npy", p_test)

rate = TARGET_POS_RATE if TARGET_POS_RATE is not None else float(y_dev.mean())
thr = prior_threshold(p_test, rate)
test_pred = (p_test >= thr).astype(int)

print(f"positive rate at 0.5 -> {np.mean(p_test >= 0.5):.3f}  (dev truth {y_dev.mean():.2f})")
print(f"final rule: top {rate:.0%} of the test scores of the {SUBMIT_WEIGHTS} weights -> class 1 | predicted positives {test_pred.mean():.3f}")

test_ids = [str(x) for x in D["test"]["ids"]]
SUBMISSION_PATH = OUT_DIR / "submission_exp3_swa_decoder_lorma_plus.json"
json.dump([{"id": cid, "pred": int(pr)} for cid, pr in zip(test_ids, test_pred)], open(SUBMISSION_PATH, "w"), indent=2)

common = dict(backbone_config=asdict(P1_CFG), backbone="phase1 SWA decoder (frozen)", K=K,
              peft=dict(method="LoRMA+", variant=LORMA_VARIANT, r=LORMA_R, alpha=LORMA_ALPHA,
                        targets=list(LORMA_TARGETS), swap_ab=LORMA_SWAP_AB,
                        implementation="scratch, Assignment-4 Q4.4 (no peft / loralib)"),
              resid_dropout=RESID_DROPOUT, head_dropout=HEAD_DROPOUT,
              preprocessing="whitespace-normalised, lower-cased, space tokens removed, last 4096 tokens, 4 chunks of 1022",
              window=WIN, special_ids=dict(pad=PAD_ID, unk=UNK_ID, bos=BOS_ID, eos=EOS_ID))
MODEL_PATH = OUT_DIR / "exp3_swa_decoder_lorma_plus_model.pt"
torch.save(dict(lorma_head_state_dict=sub_state, trainable_keys=TRAINABLE_KEYS,
                note="load the Phase-1 best_model_swa.pt backbone, attach_lorma, then load these keys with strict=False",
                trained_on=f"train only; one {E_MAX}-epoch run; dev used for monitoring only",
                weights_used=SUBMIT_WEIGHTS, step=sub_step, dev_auc=sub_auc,
                final_rule=f"top {rate:.0%} of the test scores -> class 1", **common), MODEL_PATH)
print("saved:", SUBMISSION_PATH); print("saved:", MODEL_PATH)

positive rate at 0.5 -> 0.370  (dev truth 0.50)
final rule: top 50% of the test scores of the final weights -> class 1 | predicted positives 0.502
saved: /content/drive/MyDrive/phase4_exp3_swa_decoder_lorma_plus/submission_exp3_swa_decoder_lorma_plus.json
saved: /content/drive/MyDrive/phase4_exp3_swa_decoder_lorma_plus/exp3_swa_decoder_lorma_plus_model.pt


## 12. Validate the submission, check the merged transform, report cost / memory

`merge_lorma_weights` folds $\left(I+\tfrac{\alpha}{r}BA\right)$ into each frozen projection. The merged model must give the same predictions as the re-ordered one (up to float noise) and then runs at exactly Phase-1 cost — LoRMA$_+$ is mergeable, like LoRA, which is the point of part 3 of Q4.4.

In [14]:
sub = json.load(open(SUBMISSION_PATH))
assert isinstance(sub, list) and len(sub) == len(test_ids) == 1517
assert [r["id"] for r in sub] == test_ids and len({r["id"] for r in sub}) == len(sub)
assert all(set(r) == {"id", "pred"} and type(r["pred"]) is int and r["pred"] in (0, 1) for r in sub)
print("submission OK:", len(sub), "predictions | class counts:", {c: sum(r["pred"] == c for r in sub) for c in (0, 1)})

# merge check on a small slice of the test set
probe = test_items[:64]
p_unmerged, _ = predict_proba(mF, probe)
merge_lorma_weights(mF)
p_merged, _ = predict_proba(mF, probe)
print(f"merge check | max |delta P(1)| over 64 documents: {np.abs(p_unmerged - p_merged).max():.2e}")

rep = lorma_param_report(mF)
n_total = rep["total_params"]
n_emb = mF.backbone.token_embedding.weight.numel()
avg_w = float(np.mean([n_windows("test", i) for i in range(D["test"]["n"])]))
peak = R["peak_mem_gb"]
cost = dict(total_params=n_total, embedding_params=n_emb,
            lorma_params=rep["lorma_params"], head_params=rep["head_params"],
            trainable_params=rep["trainable_params"],
            trainable_percent=100 * rep["trainable_fraction"],
            adapter_fp32_size_mb=rep["trainable_params"] * 4 / 1024**2,
            fp32_size_mb=n_total * 4 / 1024**2, fp16_size_mb=n_total * 2 / 1024**2,
            avg_windows_per_test_doc=avg_w,
            forward_gflops_per_avg_doc=2 * (n_total - n_emb) * WIN * avg_w / 1e9,
            peak_train_gpu_gb=peak, memory_cap_gb=MEM_CAP_GB)
json.dump(cost, open(OUT_DIR / "cost_exp3_swa_lorma_plus.json", "w"), indent=1)
for k, v in cost.items():
    print(f"  {k:<28}{v:>16,.2f}" if isinstance(v, float) else f"  {k:<28}{v:>16,}")
assert peak <= MEM_CAP_GB, "peak memory above the cap"
print(f"\npeak training memory {peak:.1f} GB <= {MEM_CAP_GB} GB cap (fits a free 15 GB T4)")
print("GPU used for this run:", gpu_name)
for p_ in sorted(OUT_DIR.glob("*")):
    if p_.is_file():
        print(f"  {p_.name:<46}{p_.stat().st_size/1024**2:>9.1f} MB")

submission OK: 1517 predictions | class counts: {0: 756, 1: 761}
merge check | max |delta P(1)| over 64 documents: 8.33e-03
  total_params                     140,022,018
  embedding_params                  38,400,000
  lorma_params                         524,288
  head_params                          409,090
  trainable_params                     933,378
  trainable_percent                       0.67
  adapter_fp32_size_mb                    3.56
  fp32_size_mb                          534.14
  fp16_size_mb                          267.07
  avg_windows_per_test_doc                3.26
  forward_gflops_per_avg_doc            678.83
  peak_train_gpu_gb                       4.98
  memory_cap_gb                          14.00

peak training memory 5.0 GB <= 14.0 GB cap (fits a free 15 GB T4)
GPU used for this run: NVIDIA RTX PRO 6000 Blackwell Server Edition
  cost_exp3_swa_lorma_plus.json                       0.0 MB
  exp3_swa_decoder_lorma_plus_model.pt                3.6 MB
  phase1